# 워크숍 2 — 인식

**예상 소요 시간:** 약 90분  
**선수 학습:** 워크숍 1 완료, `opencv-python`, `Pillow`, `transformers`, `torch` 설치

이 워크숍에서는 OpenCV 이미지 처리, HSV 색상 검출, 중심점으로 물체 각도 추정, Depth Anything v2 단안 깊이 추정, 구조화된 관찰을 반환하는 `perceive()` 함수를 학습합니다.

> ⚠️ **시작하기 전에:** 이전 워크숍의 시뮬레이터 창을 모두 닫으세요. 시뮬레이션은 저장되지 않으며 노트북마다 새 로봇을 만듭니다. 뷰어 탭을 두 개 열면 혼동될 수 있습니다.

## 섹션 1 — 빠른 설정

워크숍 1과 같은 연결 방식입니다. 자세한 설명은 워크숍 1을 참고하세요.

In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv opencv-python Pillow transformers torch matplotlib

In [ ]:
# 워크숍 1과 동일한 설정 — 자세한 설명은 해당 노트북을 참고하세요
import asyncio
import os
import time
import math

import cv2
import numpy as np
import PIL.Image
import matplotlib.pyplot as plt
from IPython.display import display, Image

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── 키 로더: 먼저 .env를 확인한 다음 Colab Secrets를 확인합니다 ────────────────────────
def _load_keys():
    # 모드 B: 로컬 .env 파일
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # 모드 A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for Section 7)'}")

In [ ]:
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop2-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> 뷰어 URL을 **Google Chrome**에서 열고 장면이 로드될 때까지 기다린 뒤 아래 셀을 실행하세요.

In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """뷰어가 접속하고 스킬을 사용할 수 있을 때까지 주기적으로 확인합니다."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")

**복습:** `session.get_vision('pov')`는 로봇의 1인칭 카메라에서 원시 JPEG 바이트를 반환합니다. 이번에는 단순히 표시하는 대신 컴퓨터 비전으로 처리합니다.

In [ ]:
# 도우미: 프레임을 원시 JPEG 바이트로 가져옵니다 (아래 인식 셀에는 바이트가 필요합니다.
# 워크숍 1의 screenshot()은 표시하지만 이 함수는 바이트를 반환합니다.
async def grab_frame():
    return await session.get_vision("pov")

## 섹션 2 — OpenCV로 원시 이미지 처리하기

In [ ]:
# 로봇 카메라 이미지를 가져옵니다
jpeg = await session.get_vision("pov")

# JPEG 바이트를 OpenCV용 BGR numpy 배열로 디코딩합니다
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
print(f"Image shape: {img.shape}  (height x width x channels)")

# 표시: PIL 사용(정확한 색상을 위해 BGR→RGB 변환)
pil_img = PIL.Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
display(pil_img)

# 중앙 관심 영역(가운데 1/3)을 잘라냅니다
h, w = img.shape[:2]
roi = img[h//3 : 2*h//3, w//3 : 2*w//3]
print("Center ROI:")
display(PIL.Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)))

## 섹션 3 — HSV 임곗값을 이용한 색상 분할

카메라는 BGR 이미지를 생성합니다. 조명이 달라져도 색상을 안정적으로 분리하기 위해 **HSV**(색상-채도-명도) 공간으로 변환하고 색상별 범위를 적용합니다. 빨강은 색상환의 양 끝(H≈0, H≈170)에 걸치므로 두 범위를 OR로 합쳐야 합니다.

In [ ]:
jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# 큐브 색상별 HSV 범위
COLOR_RANGES = {
    "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
               (np.array([160,50, 50]), np.array([180, 255, 255]))],
    "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
    "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
    "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
}

annotated = img.copy()

for color, ranges in COLOR_RANGES.items():
    # 이진 마스크를 만듭니다
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv, lo, hi)

    # 찾기: 블롭의 윤곽선을 찾습니다
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 그리기: 크기가 충분한 블롭에 경계 상자를 그립니다
    BGR = {"red": (0,0,255), "green": (0,200,0), "blue": (255,0,0), "yellow": (0,200,200)}
    for cnt in contours:
        if cv2.contourArea(cnt) > 200:
            x, y, bw, bh = cv2.boundingRect(cnt)
            cv2.rectangle(annotated, (x, y), (x+bw, y+bh), BGR[color], 2)
            cv2.putText(annotated, color, (x, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, BGR[color], 1)

print("Annotated image with detected color blobs:")
display(PIL.Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))

## 섹션 4 — 중심점 및 각도 오프셋 추정

탐색하려면 물체가 있는지뿐 아니라 화면의 *어디*에 있는지 알아야 합니다. 이미지 중심에서의 가로 오프셋은 로봇이 돌아야 할 각도와 대응합니다.

- **중심점:** 이미지 모멘트로 `(cx, cy)=(m10/m00, m01/m00)`
- **각도 오프셋:** `(cx-W/2)/(W/2)*HFOV/2` — 양수면 화면 중심 오른쪽

In [ ]:
HFOV_HALF_DEG = 30.0  # Half of horizontal field of view (~60° total for this camera)

jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
h, w = img.shape[:2]
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

print("Color detections (from camera):")
for color, ranges in COLOR_RANGES.items():
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv, lo, hi)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    significant = [c for c in contours if cv2.contourArea(c) > 200]
    if not significant:
        continue
    largest = max(significant, key=cv2.contourArea)
    M = cv2.moments(largest)
    if M["m00"] == 0:
        continue
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    angle_offset = (cx - w/2) / (w/2) * HFOV_HALF_DEG
    area = int(cv2.contourArea(largest))
    print(f"  {color}: centroid=({cx},{cy}), angle_offset={angle_offset:+.1f}°, area={area}px²")

# scene_state의 실제 기준값과 비교해 검증합니다
print("\nGround truth (from scene_state):")
scene = await session.state.get("scene_state")
status = await session.state.get("robot_status")
rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
yaw = status.robot.pose.yaw_deg
for eid, e in scene.entities.items():
    if eid.startswith("cube_") and e.visible:
        color = e.state.get("color", "?")
        ex, ey = e.pose.position[0], e.pose.position[1]
        bearing = math.degrees(math.atan2(ey-ry, ex-rx))
        rel_angle = (bearing - yaw + 180) % 360 - 180  # relative to robot heading
        dist = math.hypot(ex-rx, ey-ry)
        print(f"  {eid} ({color}): scene_angle={rel_angle:+.1f}°, dist={dist:.1f}m")

## 섹션 5 — Depth Anything v2를 이용한 단안 깊이

[Depth Anything v2](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf)는 한 장의 RGB 이미지에서 상대 깊이 맵을 만드는 가벼운 단안 깊이 추정 모델입니다.

**참고:** 처음 실행할 때 약 100MB 모델 가중치를 다운로드합니다. 이후에는 캐시를 사용해 CPU에서도 프레임당 약 0.5초로 빠르게 실행됩니다.

In [ ]:
from transformers import pipeline as hf_pipeline

print("Loading Depth Anything v2 (first run downloads ~100MB)...")
depth_pipe = hf_pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
)
print("Model loaded.")

In [ ]:
jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
pil_img = PIL.Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

# 깊이 추정을 실행합니다
result = depth_pipe(pil_img)
depth_map = np.array(result["depth"])

print(f"Depth map shape: {depth_map.shape}")
print(f"Depth range: {depth_map.min():.2f} – {depth_map.max():.2f}")

# 나란히 시각화합니다
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(pil_img); ax1.set_title("Camera image"); ax1.axis("off")
im = ax2.imshow(depth_map, cmap="plasma")
ax2.set_title("Depth map (brighter = closer)"); ax2.axis("off")
plt.colorbar(im, ax=ax2, fraction=0.046)
plt.tight_layout()
plt.show()

# 세 픽셀 위치에서 깊이를 읽습니다
h, w = depth_map.shape
samples = [(h//2, w//4, "left"), (h//2, w//2, "center"), (h//2, 3*w//4, "right")]
print("\nDepth samples:")
for row, col, label in samples:
    print(f"  {label} ({col},{row}): depth={depth_map[row, col]:.2f}")

## 섹션 6 — 색상 검출과 깊이 결합하기

검출한 각 색상 블롭의 중심점에서 깊이 맵 값을 읽으면 물체를 대략적인 거리 순서로 정렬할 수 있습니다.

In [ ]:
jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
pil_img = PIL.Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, w = img.shape[:2]

depth_result = depth_pipe(pil_img)
depth_map = np.array(depth_result["depth"])

print("Color blobs with depth scores:")
detections = []
for color, ranges in COLOR_RANGES.items():
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv, lo, hi)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        continue
    largest = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest)
    if area < 200:
        continue
    M = cv2.moments(largest)
    if M["m00"] == 0:
        continue
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    # 깊이 맵 범위 안으로 제한합니다
    dy = min(max(cy, 0), depth_map.shape[0]-1)
    dx = min(max(cx, 0), depth_map.shape[1]-1)
    depth_score = float(depth_map[dy, dx])
    detections.append((color, area, depth_score))

# 깊이 점수로 정렬합니다(Depth Anything에서는 값이 클수록 가까움)
for color, area, depth in sorted(detections, key=lambda x: -x[2]):
    print(f"  {color}: area={int(area)}px², depth_score={depth:.2f}")

## 섹션 7 — `perceive()` 함수

지금까지의 기능을 하나로 묶어 카메라 프레임에서 구조화된 관찰 결과를 반환하는 비동기 함수를 만듭니다.

In [ ]:
async def perceive(session):
    """Return dict of {color: {angle_deg, blob_area}} for all visible cubes."""
    jpeg = await session.get_vision("pov")
    img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    COLOR_RANGES_LOCAL = {
        "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
                   (np.array([160,50, 50]), np.array([180, 255, 255]))],
        "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
        "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
        "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
    }

    result = {}
    for color, ranges in COLOR_RANGES_LOCAL.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, lo, hi)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        if area < 200:
            continue
        M = cv2.moments(largest)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        angle_deg = (cx - w / 2) / (w / 2) * 30.0  # HFOV/2 = 30°
        result[color] = {"angle_deg": round(angle_deg, 1), "blob_area": int(area)}

    return result


# 실행 예시
obs = await perceive(session)
print("perceive() output:")
for color, data in obs.items():
    print(f"  {color}: angle={data['angle_deg']:+.1f}°, area={data['blob_area']}px²")

---
## 연습 문제

### 연습 문제 1 — 이미지에 시간 표시하기

`cv2.putText`를 사용해 로봇 POV 왼쪽 위에 현재 시간을 표시하세요.

**힌트:** `cv2.putText(img_copy, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)`

In [ ]:
## 연습 문제 1: POV에 시간 표시하기
import datetime

# 여기에 코드를 작성하세요

# 예상 결과: image displayed with white timestamp text in the top-left corner,
# e.g. "2024-06-17 14:32:01"

### 연습 문제 2 — 초록 큐브 검출하고 상자 그리기

HSV 임곗값(H: 40–80, S: 50–255, V: 50–255)으로 초록 큐브를 분할하세요. 가장 큰 윤곽선을 찾고 경계 상자를 그린 뒤 중심점 좌표를 출력하세요.

In [ ]:
## 연습 문제 2: 초록 큐브를 검출하고 경계 상자와 중심점 출력
# 여기에 코드를 작성하세요

# 예상 결과:
# Green cube centroid: (cx=312, cy=198)
# [image with green rectangle drawn around the cube]

### 연습 문제 3 — 카메라 각도와 장면 상태 각도 비교하기

보이는 각 큐브에 대해 (1) 섹션 4처럼 카메라 중심점 각도, (2) `scene_state`와 `robot_status`를 이용한 실제 기준 각도(robot yaw + 큐브까지의 atan2), (3) 두 값의 오차를 계산해 출력하세요. 카메라 추정값은 실제 기준과 얼마나 비슷한가요?

In [ ]:
## 연습 문제 3: 카메라 각도 추정값과 scene_state 실제 기준값 비교
# 여기에 코드를 작성하세요

# 예상 출력(예시):
# red:   camera=+12.3°,  scene=+10.8°,  error=1.5°
# green: camera=-5.1°,   scene=-6.2°,   error=1.1°

### 연습 문제 4 — 검출된 블롭 중심점에서 깊이 읽기

현재 프레임에 Depth Anything을 실행하고 각 색상 블롭의 중심점에서 깊이 값을 읽으세요. 가까운 순서부터(`depth_score`가 높은 순서) 출력하세요.

In [ ]:
## 연습 문제 4: Depth Anything 실행 후 블롭 중심점 값 읽기
# 여기에 코드를 작성하세요

# 예상 출력(가까운 순서):
# blue:   depth_score=0.82
# red:    depth_score=0.61
# green:  depth_score=0.43

### 연습 문제 5 — `find_nearest_cube(jpeg_bytes)`

다음을 수행하는 `find_nearest_cube(jpeg_bytes) -> (color, angle_offset_deg)`를 구현하세요.
- JPEG 바이트에 HSV 분할 실행
- 모든 색상 중 면적이 가장 큰 블롭 찾기(가장 큰 블롭 ≈ 가장 가까운 큐브)
- 해당 블롭의 `(color, angle_offset_deg)` 반환
- 블롭이 없으면 `(None, None)` 반환

In [ ]:
## 연습 문제 5: find_nearest_cube
def find_nearest_cube(jpeg_bytes):
    # 여기에 코드를 작성하세요
    pass

# 테스트
jpeg = await session.get_vision("pov")
color, angle = find_nearest_cube(jpeg)
print(f"Nearest cube: {color} at {angle}°")

# 예상 출력(예시): Nearest cube: red at +12.3°
# 화면에 큐브가 없으면 (None, None)을 반환합니다

### 연습 문제 6(도전) — 빨간 큐브가 중앙에 올 때까지 회전하기

`find_nearest_cube` 또는 `perceive`를 반복 호출하세요. 프레임에서 빨간 큐브가 중앙 ±10° 안인지 확인하고, 아니라면 `set_velocity(wz=0.3, vx=0.1, duration_s=1.0)`으로 조금 회전합니다. 중앙에 오거나 12단계가 지나면 멈추고 매 단계의 각도 오프셋을 출력하세요.

In [ ]:
## 연습 문제 6 (stretch): 빨간 큐브가 중앙에 올 때까지 회전
# 여기에 코드를 작성하세요

# Expected output:
# 단계 1: red at +22.1° — 오른쪽으로 회전
# 단계 2: red at +14.3° — 오른쪽으로 회전
# 단계 3: red at +6.8°  — 중앙 정렬 완료!
# (or: 빨간색이 보이지 않음 — 찾기 위해 회전)

---
## 정리

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")